In [7]:
%pip install numpy
%pip install plyfile

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
##Morton Code
from plyfile import PlyElement, PlyData
import numpy as np

ply = PlyData.read("Goat skull.ply")

v = ply['vertex']

# Extract as NumPy arrays
points = np.vstack((v['x'], v['y'], v['z'])).T
colors = np.vstack((v['red'], v['green'], v['blue'])).T.astype(np.uint8)

print(points.shape, colors.shape)

def normalize_to_int(points, bits=10):
    max_val = (1 << bits) - 1
    pts = points - points.min(axis=0)
    pts = pts / pts.max(axis=0)
    return (pts * max_val).astype(np.uint32)

def part1by2(n):
    n &= 0x3ff
    n = (n | n << 16) & 0x30000ff
    n = (n | n << 8)  & 0x300f00f
    n = (n | n << 4)  & 0x30c30c3
    n = (n | n << 2)  & 0x9249249
    return n

def morton3D(x, y, z):
    return (part1by2(z) << 2) | (part1by2(y) << 1) | part1by2(x)

int_pts = normalize_to_int(points, bits=10)

morton = np.array([
    morton3D(x, y, z) for x, y, z in int_pts
], dtype=np.uint64)

order = np.argsort(morton)

points_sorted = points[order]
colors_sorted = colors[order]

# Build structured array for vertices
vertex_data = np.empty(points_sorted.shape[0],
                       dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                              ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])

vertex_data['x'] = points_sorted[:,0]
vertex_data['y'] = points_sorted[:,1]
vertex_data['z'] = points_sorted[:,2]
vertex_data['red']   = colors_sorted[:,0]
vertex_data['green'] = colors_sorted[:,1]
vertex_data['blue']  = colors_sorted[:,2]

vertex_el = PlyElement.describe(vertex_data, 'vertex')

# Write output
out = PlyData([vertex_el], text=False)
out.write("Goat Morton.ply")

(983599, 3) (983599, 3)


In [15]:
##Shuffle
from plyfile import PlyData, PlyElement
import numpy as np

ply = PlyData.read("Goat skull.ply")
vertex = ply['vertex']

# Extract all properties as a structured array
data = vertex.data
data = np.array(data)

# Shuffle rows
np.random.shuffle(data)

# Write shuffled PLY
el = PlyElement.describe(data, 'vertex')
PlyData([el], text=ply.text).write("Goat Shuffled.ply")

In [8]:
##Morton Shuffle
from plyfile import PlyElement, PlyData
import numpy as np
import random

ply = PlyData.read("Goat skull.ply")

v = ply['vertex']

# Extract as NumPy arrays
points = np.vstack((v['x'], v['y'], v['z'])).T
colors = np.vstack((v['red'], v['green'], v['blue'])).T.astype(np.uint8)

print(points.shape, colors.shape)

def normalize_to_int(points, bits=10):
    max_val = (1 << bits) - 1
    pts = points - points.min(axis=0)
    pts = pts / pts.max(axis=0)
    return (pts * max_val).astype(np.uint32)

def part1by2(n):
    n &= 0x3ff
    n = (n | n << 16) & 0x30000ff
    n = (n | n << 8)  & 0x300f00f
    n = (n | n << 4)  & 0x30c30c3
    n = (n | n << 2)  & 0x9249249
    return n

def morton3D(x, y, z):
    return (part1by2(z) << 2) | (part1by2(y) << 1) | part1by2(x)

int_pts = normalize_to_int(points, bits=10)

morton = np.array([
    morton3D(x, y, z) for x, y, z in int_pts
], dtype=np.uint64)

order = np.argsort(morton)

points_sorted = points[order]
colors_sorted = colors[order]

batch_size = 128
N = points_sorted.shape[0]
num_batches = (N + batch_size - 1) // batch_size 

points_batches = [
    points_sorted[i*batch_size:(i+1)*batch_size]
    for i in range(num_batches)
]

colors_batches = [
    colors_sorted[i*batch_size:(i+1)*batch_size]
    for i in range(num_batches)
]

indices = list(range(num_batches))
random.shuffle(indices)

points_shuffled = [points_batches[i] for i in indices]
colors_shuffled = [colors_batches[i] for i in indices]

points_final = np.vstack(points_shuffled)
colors_final = np.vstack(colors_shuffled)

# Build structured array for vertices
vertex_data = np.empty(points_sorted.shape[0],
                       dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                              ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])

vertex_data['x'] = points_final[:,0]
vertex_data['y'] = points_final[:,1]
vertex_data['z'] = points_final[:,2]
vertex_data['red']   = colors_final[:,0]
vertex_data['green'] = colors_final[:,1]
vertex_data['blue']  = colors_final[:,2]

vertex_el = PlyElement.describe(vertex_data, 'vertex')

# Write output
out = PlyData([vertex_el], text=False)
out.write("Goat Morton Shuffle.ply")

(983599, 3) (983599, 3)


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def lebesgue2d(num_points):
    # Function to calculate Lebesgue coordinates
    def lebesgue_coords(z):
        coords = {'x': 0, 'y': 0}
        shift_mask = 0x55555555  # Mask for even bits
        
        # Mask out even bits for x, odd bits for y
        coords['x'] = z & shift_mask
        coords['y'] = (z & 0xaaaaaaaa) >> 1  # Shift the odd bits to y
        
        shift_mask = 0xfffffffc  # Adjust the mask to handle 2-bit wide groups
        for i in range(32):
            # Compress bits by extracting the top bits and shifting them down one
            x_upper = (coords['x'] & shift_mask) >> 1
            y_upper = (coords['y'] & shift_mask) >> 1
            
            # Clear out the top bits from x and reintroduce the shifted bits
            coords['x'] = x_upper | (coords['x'] & ~shift_mask)
            coords['y'] = y_upper | (coords['y'] & ~shift_mask)
            
            shift_mask <<= 1  # Reduce the mask size, processing pairs of bits
        
        return coords
    
    # Function to compute the Lebesgue coordinates for a range of z values
    def lebesgue(z_values):
        x_values = []
        y_values = []
        
        for z in z_values:
            coords = lebesgue_coords(z)
            x_values.append(coords['x'])
            y_values.append(coords['y'])
        
        return pd.DataFrame({'x': x_values, 'y': y_values})
    
    z_values = range(0, num_points)  
    lebesgue_points = pd.DataFrame({'z': z_values})
    
    # Apply the lebesgue function to calculate 'x' and 'y' coordinates
    lebesgue_points[['x', 'y']] = lebesgue(z_values)
    
    # Print the resulting DataFrame
    print(lebesgue_points)
    
    # Plot the data with lines connecting the points
    plt.figure(figsize=(6, 6))
    
    # Plot the 'x' and 'y' points as a line
    plt.plot(lebesgue_points['x'], lebesgue_points['y'], marker='o', color='b', linestyle='-', markersize=4)
    
    # Adding labels and title
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Lebesgue Curve: Connecting Points')
    
    # Show grid for better readability
    plt.grid(True)
    
    # Display the plot
    plt.show()
    
    # Calculate the coordinate distances
    lebesgue_points['coord_distance'] = np.sqrt(
        (lebesgue_points['x'] - lebesgue_points['x'].shift(1))**2 + 
        (lebesgue_points['y'] - lebesgue_points['y'].shift(1))**2
    )
    
    # Remove rows with NaN values (first row will have NaN distance)
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Calculate the mean distance
    mean_distance = lebesgue_locality['coord_distance'].mean()
    
    # Print the result
    print(f"Mean distance: {mean_distance}")
    
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Plot the coordinate distances
    plt.figure(figsize=(12, 6))
    plt.plot(lebesgue_locality['z'], lebesgue_locality['coord_distance'], marker='o', color='b', linestyle='-', markersize=4)
    plt.xlabel('z')
    plt.ylabel('Distance Between Points')
    plt.title('Distance Between Consecutive Points')
    plt.grid(True)
    plt.show()

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def lebesgue3d(num_points):
    # Function to calculate Lebesgue coordinates
    def lebesgue_coords(w):
        coords = {'x': 0, 'y': 0, 'z':0}
        
        
        coords['x'] = w & 0x49249249
        coords['y'] = (w & 0x92492492) >> 1
        coords['z'] = (w & 0x24924924) >> 1
        
        shift_mask = 0xfffffff8  # Adjust the mask to handle 2-bit wide groups
        for i in range(32):
            # Compress bits by extracting the top bits and shifting them down one
            x_upper = (coords['x'] & shift_mask) >> 1
            y_upper = (coords['y'] & shift_mask) >> 1
            z_upper = (coords['z'] & shift_mask) >> 1
            
            # Clear out the top bits from x and reintroduce the shifted bits
            coords['x'] = x_upper | (coords['x'] & ~shift_mask)
            coords['y'] = y_upper | (coords['y'] & ~shift_mask)
            coords['z'] = z_upper | (coords['z'] & ~shift_mask)
            
            shift_mask <<= 1  # Reduce the mask size, processing pairs of bits
        
        return coords
    
    # Function to compute the Lebesgue coordinates for a range of z values
    def lebesgue(w_values):
        x_values = []
        y_values = []
        z_values = []
        
        for w in w_values:
            coords = lebesgue_coords(w)
            x_values.append(coords['x'])
            y_values.append(coords['y'])
            z_values.append(coords['z'])
        
        return pd.DataFrame({'x': x_values, 'y': y_values, 'z': z_values})
    
    w_values = range(0, num_points)  
    lebesgue_points = pd.DataFrame({'w': w_values})
    
    # Apply the lebesgue function to calculate 'x' and 'y' coordinates
    lebesgue_points[['x', 'y', 'z']] = lebesgue(w_values)
    
    # Print the resulting DataFrame
    print(lebesgue_points)

    # Calculate the coordinate distances
    lebesgue_points['coord_distance'] = np.sqrt(
        (lebesgue_points['x'] - lebesgue_points['x'].shift(1))**2 + 
        (lebesgue_points['y'] - lebesgue_points['y'].shift(1))**2 + 
        (lebesgue_points['z'] - lebesgue_points['z'].shift(1))**2
    )
    
    # Remove rows with NaN values (first row will have NaN distance)
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # Calculate the mean distance
    mean_distance = lebesgue_locality['coord_distance'].mean()
    
    # Print the result
    print(f"Mean distance: {mean_distance}")
    
    lebesgue_locality = lebesgue_points.dropna(subset=['coord_distance'])
    
    # 3D Plotting
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the 3D scatter plot of the points (x, y, z)
    ax.scatter(lebesgue_locality['x'], lebesgue_locality['y'], lebesgue_locality['z'], c='b', marker='o', label='Points')
    
    # Optional: Connect consecutive points with lines to show distances
    for i in range(1, len(lebesgue_locality)):
        ax.plot(
            [lebesgue_locality['x'].iloc[i-1], lebesgue_locality['x'].iloc[i]],
            [lebesgue_locality['y'].iloc[i-1], lebesgue_locality['y'].iloc[i]],
            [lebesgue_locality['z'].iloc[i-1], lebesgue_locality['z'].iloc[i]],
            color='r', linestyle='-', linewidth=0.5  # Red lines connecting consecutive points
        )
    
    # Labels and title
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('3D Plot of Lebesgue Points')
    
    # Show the plot
    plt.show()
    
    # --- 2. 2D Line Plot for Distance Progression (Separate Figure) ---
    fig_2d = plt.figure(figsize=(12, 8))
    
    # Create a 2D axis for the second plot
    ax_2d = fig_2d.add_subplot(111)
    
    # Plot the 2D line plot of distances
    ax_2d.plot(lebesgue_locality['w'], lebesgue_locality['coord_distance'], color='r', label='Distance Progression')
    
    # Labels and title for the 2D plot
    ax_2d.set_xlabel('w')
    ax_2d.set_ylabel('Distance')
    ax_2d.set_title('2D Plot of Distance Progression')
    ax_2d.grid(True)
    
    # Display the 2D plot
    plt.show()

In [ ]:
lebesgue3d(16777216)

                 w     x     y     z
0                0     0     0     0
1                1     1     0     0
2                2     0     1     0
3                3     1     1     0
4                4     0     0     2
...            ...   ...   ...   ...
16777211  16777211  3509  3509  5848
16777212  16777212  3508  3508  5850
16777213  16777213  3509  3508  5850
16777214  16777214  3508  3509  5850
16777215  16777215  3509  3509  5850

[16777216 rows x 4 columns]
Mean distance: 2.1511358475076103


KeyboardInterrupt: 